In [2]:
!pip install 'pydantic[email]' #para o código rodar no notebook foi necessário fazer essa instalação pois apareciam erros de execução ao tentar validar o email

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 8.9 MB/s eta 0:00:00


In [3]:
import sys
from enum import auto, IntFlag #À partir do módulo "enum" é feita a importação da função "auto" (cujo objetivo é a atribuição de valores crescentes inteiros) e da função "IntFlag" usada com o objetivo de representar múltiplas oppções ao mesmo tempo
from typing import Any #Importa Any do módulo typing, basicamente é útil para aceitar qualquer valor

from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    SecretStr,
    ValidationError,
) #Aqui está importando da biblioteca pydantic "BaseModel" (usada para criar modelos de dados validados), "EmailStr" (usado para validar email), "Field" (usada para criar regras extras de validação), "SecretStr" (usada para dados sensíveis como senhar, colocando * no lugar do valor), "ValidationError" (usada para mostrar o que deu errado)

In [4]:
class Role(IntFlag): #Aqui é demonstrado o uso do "auto" quanto da "IntFlag", demonstrando o uso de parênteses vazios em auto para receber o valor e a combinação de valor em Admin separados por | (também seria permitido o uso de &)
  Author=auto()
  Editor=auto()
  Developer=auto()
  Admin=Author| Editor| Developer

class User(BaseModel): #Aqui é criada a classe "user" utilizando BaseModel importado do pydantic para criar o modelo validado
  name: str=Field(examples=["Karol"]) #aqui é utilizado Field para documentar um exemplo
  email: EmailStr=Field( #Além do EmailStr utilizado para validar o formato de email, é utilizado Field para documentar e fornecer exemplo de uso
      examples=["example@karolreges.com"],
      description="The email adress of the user",
      frozen=True, #Esse parâmetro é utilizado para definir o email como campo imutável, particularmente acho errôneo pois existem casos onde a atualização de email se faz necessária, acredito que se enquadraria melhor em CPF ou outro tipo de ID único
  )
  password: SecretStr=Field( #SecrectStr vinda da biblioteca do pydantic funciona para mascarar com * a senha
      examples=["Password123"],
      description="The password of the user"
  )
  role: Role=Field( #a classe Role que criamos acima é chamada para definir o papel do usuário
      default=None, #aqui fica um dúvida pois foram definidos os tipos
      description="The role of the user"
  )

def validate(data: dict[str, Any]) -> None: #Essa função recebe os dados preenchidos em user e tenta validar eles com base nas regras definidas no BaseModel, se validas, o objeto é criado e impresso, caso contrário imprime mensagem de erro e detalha os problemas
  try:
    user= User.model_validate(data)
    print(user)
  except ValidationError as e:
    print("User is invalid")
    for error in e.errors():
      print(error)

def main() -> None: #na função principal criamos dois tipos de dados de usuários, um correto e outro que irá gerar erro
  good_data={
      "name": "Karol",
      "email":"karolreges@ufg.com",
      "password":"Password123",
  }
  bad_data={
      "email": "<bad_data>",
      "password": "<bad_data>"
  }
  print("TESTE DOS DADOS CORRETOS")
  validate(good_data) #aqui fazemos a validação desses dados
  print("\nTESTE DOS DADOS COM ERRO")
  validate(bad_data)

if __name__=="__main__": #algumas tentativas e erros foram feitas até perceber um erro com o espaçamento, fazendo linha ficar dentro da função main, o que não imprimia o resultado do teste
    main()


TESTE DOS DADOS CORRETOS
name='Karol' email='karolreges@ufg.com' password=SecretStr('**********') role=None

TESTE DOS DADOS COM ERRO
User is invalid
{'type': 'missing', 'loc': ('name',), 'msg': 'Field required', 'input': {'email': '<bad_data>', 'password': '<bad_data>'}, 'url': 'https://errors.pydantic.dev/2.12/v/missing'}
{'type': 'value_error', 'loc': ('email',), 'msg': 'value is not a valid email address: An email address must have an @-sign.', 'input': '<bad_data>', 'ctx': {'reason': 'An email address must have an @-sign.'}}


In [5]:
import enum
import hashlib #importa o módulo hashlib que transforma entradas em hashes, usado normalmente em senhas
import re #importa o módulo de expressões regulares que busca, extrai, substitui ou manipula padrões de texto
from typing import Any

from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    field_validator, #validador de campo especifico
    model_validator, #validador de relação entre campos
    SecretStr,
    ValidationError,
)

#São aplicadas regras para a senha e para o nome, a senha deve ter letra maiúscula, minúscula e número e o nome deve ter apenas letra, pelo menos 2 caracteres e sem espaços
VALID_PASSWORD_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")
VALID_NAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")

class Role(enum.IntFlag): #mais uma vez é criada a classe dos papeis só que agora atribuindo valores para cada papel
  Author=1
  Editor=2
  Admin=4
  SuperAdmin=8

class User (BaseModel):
  name: str=Field(examples=["Karol"])
  email:EmailStr=Field(
      examples=["example@karolreges.com"],
      description="The email adress of the user",
      frozen=True,
  )
  password: SecretStr=Field(
      examples=["Password123"],
      description="The password of the user"
  )
  role: Role=Field(
      default=None,
      description="The role of the user",
      examples=[1, 2, 4, 8] #como agora os tipos recebem um valor, eles também entram aqui como exemplos
  )

  @field_validator("name") #é criado um validador para "name" que irá guardar e comparar o valor dessa string com o REGEX criado para "name", caso esteja fora do padrão definido, será emitido um aviso de erro
  @classmethod
  def validate_name(cls, v: str)->str:
    if not VALID_NAME_REGEX.match(v):
      raise ValueError(
          "Name is invalid, must contais only letters and be at least 2 characters long"
      )
    return v

  @field_validator("role", mode="before") #aqui temos o validador dos papeis, como é possível receber como entrada um inteiro, uma string ou o papel já definido, é criada uma tabela de conversão que independete da entrada irá fornecer ao final um valor com o papel
  @classmethod
  def validate_role(cls, v:int | str | Role) -> Role:
    op= {int: lambda x: Role(x), str: lambda x: Role[x], Role: lambda x: x}
    try:
      return op[type(v)](v)
    except (KeyError, ValueError): #em caso de retornar um valor diferente dos papeis definidos, será emitido um aviso derro
      raise ValueError(
          f"Role is invalid, please use one of the following: {', '.join([x.name for x in Role])}"
      )

  @model_validator(mode="before") #Aqui é realizada uma validação do usuário em geral e pelo que entedi o before força que essa validação ocorra a nível de formulário, antes de fazer a validação do pydantic
  @classmethod
  def validate_user(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "name" not in v or "password" not in v: #o primeiro if verifica se nome e senha foram preenchidos, sendo ambos campos obrigatórios
            raise ValueError("Name and password are required")
        if v["name"].casefold() in v["password"].casefold(): #faz a verificação se contém o nome dentro da senha, me parece uma técnica para evitar senhas pouco seguras
            raise ValueError("Password cannot contain name")
        if not VALID_PASSWORD_REGEX.match(v["password"]): #por fim é feita a verificação do REGEX se a senha segue as regras estabelecidas (me lembrou a validação que contém em alguns logins mostrando que falta algo na senha antes de fazer a requisição)
            raise ValueError(
                "Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number"
            )
        v["password"] = hashlib.sha256(v["password"].encode()).hexdigest() #ocorre a transformação da senha em bytes, depois em hexadeciaml e ela recebe hashes para ocultar o valor
        return v

def validate(data: dict[str, Any]) -> None: #é criada a função que irá rodar todas as validações que criamos acima
    try:
        user = User.model_validate(data) #se as validações derem certo, o usuário é impresso
        print(user)
    except ValidationError as e: #caso surja algum erro, é impressa a mensagem de erro e qual erro existe
        print("User is invalid:")
        print(e)

def main() -> None: #aqui temops a função principal que irá inserir os dados e fazer a validação deles
    test_data = dict(
        good_data={
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Password123",
            "role": "Admin",
        },
        bad_role={
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Password123",
            "role": "Programmer",
        },
        bad_data={
            "name": "Arjan",
            "email": "bad email",
            "password": "bad password",
        },
        bad_name={
            "name": "Arjan<-_->",
            "email": "example@arjancodes.com",
            "password": "Password123",
        },
        duplicate={
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Arjan123",
        },
        missing_data={
            "email": "<bad data>",
            "password": "<bad data>",
        },
    )

    for example_name, data in test_data.items():
        print(example_name)
        validate(data)
        print()


if __name__ == "__main__":
    main()

good_data
name='Arjan' email='example@arjancodes.com' password=SecretStr('**********') role=<Role.Admin: 4>

bad_role
User is invalid:
1 validation error for User
role
  Value error, Role is invalid, please use one of the following: Author, Editor, Admin, SuperAdmin [type=value_error, input_value='Programmer', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

bad_data
User is invalid:
1 validation error for User
  Value error, Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number [type=value_error, input_value={'name': 'Arjan', 'email'...ssword': 'bad password'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

bad_name
User is invalid:
1 validation error for User
name
  Value error, Name is invalid, must contais only letters and be at least 2 characters long [type=value_error, input_value='Arjan<-_->', input_type=str]
    For further information visit https://

In [6]:
import enum
import hashlib
import re
from typing import Any, Self
from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    field_serializer, #Tanto ele quanto o model_serializer atuam na saída de dados, é utilizado para alterar um campo específico
    field_validator,
    model_serializer, #utilizado para montar respostas customizadas
    model_validator,
    SecretStr,
)

VALID_PASSWORD_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")
VALID_NAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")


class Role(enum.IntFlag):
    User = 0 #A novidade aqui vai ser o user com valor zero, o que nos indica que não possui permissão alguma
    Author = 1
    Editor = 2
    Admin = 4
    SuperAdmin = 8


class User(BaseModel):
    name: str = Field(examples=["Example"])
    email: EmailStr = Field(
        examples=["user@arjancodes.com"],
        description="The email address of the user",
        frozen=True,
    )
    password: SecretStr = Field(
        examples=["Password123"], description="The password of the user", exclude=True
    )
    role: Role = Field(
        description="The role of the user",
        examples=[1, 2, 4, 8],
        default=0,#Define o valor padrão 0, ou seja, usuário sem permissões caso não seja informado um dos outros valores da lista de papeis definidas
        validate_default=True, #Define que até o valor padrão terá que passar pelos validadores
    )

    @field_validator("name")
    def validate_name(cls, v: str) -> str:
        if not VALID_NAME_REGEX.match(v):
            raise ValueError(
                "Name is invalid, must contain only letters and be at least 2 characters long"
            )
        return v

    @field_validator("role", mode="before")
    @classmethod
    def validate_role(cls, v: int | str | Role) -> Role:
        op = {int: lambda x: Role(x), str: lambda x: Role[x], Role: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            raise ValueError(
                f"Role is invalid, please use one of the following: {', '.join([x.name for x in Role])}"
            )

    @model_validator(mode="before")
    @classmethod
    def validate_user_pre(cls, v: dict[str, Any]) -> dict[str, Any]: #é feita uma validação do usuário antes de criar o objeto usuário
        if "name" not in v or "password" not in v:
            raise ValueError("Name and password are required")
        if v["name"].casefold() in v["password"].casefold():
            raise ValueError("Password cannot contain name")
        if not VALID_PASSWORD_REGEX.match(v["password"]):
            raise ValueError(
                "Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number"
            )
        v["password"] = hashlib.sha256(v["password"].encode()).hexdigest()
        return v

    @model_validator(mode="after") #Aqui ocorre uma verificação após a criação do objeto, basicamente ele está verificando se o usuário com o papel correto (no caso de admin) é o que está tentando validar os dados
    def validate_user_post(self, v: Any) -> Self:
        if self.role == Role.Admin and self.name != "Arjan":
            raise ValueError("Only Arjan can be an admin")
        return self

    @field_serializer("role", when_used="json")
    @classmethod
    def serialize_role(cls, v) -> str:
        return v.name

    @model_serializer(mode="wrap", when_used="json") #utilizado para fazer serialização dos dados, para filtrar ou não os dados de usuário
    def serialize_user(self, serializer, info) -> dict[str, Any]:
        if not info.include and not info.exclude:
            return {"name": self.name, "role": self.role.name}
        return serializer(self)


def main() -> None: #Nessa função principal são testadas as diferentes formas de exportar os dados do usuário
    data = { #Os dados são criados
        "name": "Arjan",
        "email": "example@arjancodes.com",
        "password": "Password123",
        "role": "Admin",
    }
    user = User.model_validate(data) #valida os dados de acordo com as regras definidas
    if user:
        print(
            "The serializer that returns a dict:",
            user.model_dump(), #aqui as informações serão impressas respeitando as regras, o papel deve vir com o seu nome e número interio associado
            sep="\n",
            end="\n\n",
        )
        print(
            "The serializer that returns a JSON string:",
            user.model_dump(mode="json"), #Força a apresentar apenas o nome e o nome do papel
            sep="\n",
            end="\n\n",
        )
        print(
            "The serializer that returns a json string, excluding the role:",
            user.model_dump(exclude=["role"], mode="json"), #remove o papel e apresenta as demais informações
            sep="\n",
            end="\n\n",
        )
        print("The serializer that encodes all values to a dict:", dict(user), sep="\n")


if __name__ == "__main__":
    main()

The serializer that returns a dict:
{'name': 'Arjan', 'email': 'example@arjancodes.com', 'role': <Role.Admin: 4>}

The serializer that returns a JSON string:
{'name': 'Arjan', 'role': 'Admin'}

The serializer that returns a json string, excluding the role:
{'name': 'Arjan', 'email': 'example@arjancodes.com'}

The serializer that encodes all values to a dict:
{'name': 'Arjan', 'email': 'example@arjancodes.com', 'password': SecretStr('**********'), 'role': <Role.Admin: 4>}


Basicamente esse trecho final cria usuários, valida tudo automaticamente, armazene em memória e permita consultar via API, garantindo que tudo esteja correto. Particularmente tive mais dificuldade neste trecho final, não consegui inclusive emitir muitos comentários, precisaria estudar mais o código.

In [7]:
from datetime import datetime
from typing import Optional
from uuid import uuid4

from fastapi import FastAPI
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, EmailStr, Field, field_serializer, UUID4

app = FastAPI() #É criada uma aplicação web


class User(BaseModel):
    model_config = {
        "extra": "forbid", #impede a criação de campos extras
    }
    __users__ = [] #é criada uma lista para simular uma base de dados
    name: str = Field(..., description="Name of the user")
    email: EmailStr = Field(..., description="Email address of the user")
    friends: list[UUID4] = Field(
        default_factory=list, max_items=500, description="List of friends"
    )
    blocked: list[UUID4] = Field(
        default_factory=list, max_items=500, description="List of blocked users"
    )
    signup_ts: Optional[datetime] = Field(
        default_factory=datetime.now, description="Signup timestamp", kw_only=True
    )
    id: UUID4 = Field(
        default_factory=uuid4, description="Unique identifier", kw_only=True
    )

    @field_serializer("id", when_used="json")
    def serialize_id(self, id: UUID4) -> str:
        return str(id)


@app.get("/users", response_model=list[User])
async def get_users() -> list[User]:
    return list(User.__users__)


@app.post("/users", response_model=User) #cria os usuários
async def create_user(user: User):
    User.__users__.append(user)
    return user


@app.get("/users/{user_id}", response_model=User)
async def get_user(user_id: UUID4) -> User | JSONResponse:
    try:
        return next((user for user in User.__users__ if user.id == user_id))
    except StopIteration:
        return JSONResponse(status_code=404, content={"message": "User not found"})


def main() -> None:
    with TestClient(app) as client: #Simula requisições HTTP sem rodar servidor
        for i in range(5): #cria 5 usuários
            response = client.post(
                "/users",
                json={"name": f"User {i}", "email": f"example{i}@arjancodes.com"},
            )
            assert response.status_code == 200
            assert response.json()["name"] == f"User {i}", (
                "The name of the user should be User {i}"
            )
            assert response.json()["id"], "The user should have an id"

            user = User.model_validate(response.json())
            assert str(user.id) == response.json()["id"], "The id should be the same"
            assert user.signup_ts, "The signup timestamp should be set"
            assert user.friends == [], "The friends list should be empty"
            assert user.blocked == [], "The blocked list should be empty"

        response = client.get("/users")
        assert response.status_code == 200, "Response code should be 200"
        assert len(response.json()) == 5, "There should be 5 users"

        response = client.post(
            "/users", json={"name": "User 5", "email": "example5@arjancodes.com"}
        )
        assert response.status_code == 200
        assert response.json()["name"] == "User 5", (
            "The name of the user should be User 5"
        )
        assert response.json()["id"], "The user should have an id"

        user = User.model_validate(response.json())
        assert str(user.id) == response.json()["id"], "The id should be the same"
        assert user.signup_ts, "The signup timestamp should be set"
        assert user.friends == [], "The friends list should be empty"
        assert user.blocked == [], "The blocked list should be empty"

        response = client.get(f"/users/{response.json()['id']}")
        assert response.status_code == 200
        assert response.json()["name"] == "User 5", (
            "This should be the newly created user"
        )

        response = client.get(f"/users/{uuid4()}")
        assert response.status_code == 404
        assert response.json()["message"] == "User not found", (
            "We technically should not find this user"
        )

        response = client.post("/users", json={"name": "User 6", "email": "wrong"})
        assert response.status_code == 422, "The email address is should be invalid"


if __name__ == "__main__":
    main()

/tmp/ipykernel_435/4128769287.py:20: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  friends: list[UUID4] = Field(
/tmp/ipykernel_435/4128769287.py:23: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  blocked: list[UUID4] = Field(
